- 基本控制：串行控制
- 基本控制：并行控制
- 基本控制：条件与循环
- 精细控制：图的运行时配置
- 精细控制：map-reduce

In [ ]:
! pip install -U langgraph

定义state
****
- TypedDict: 属于Python标准库Typing模块的一部分，仅提供静态类型检查，运行时不执行验证
- Pydantic: 第三方库，需要单独安装，提供运行时数据验证和序列化功能

In [6]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict


# 定义节点间通讯的消息格式
class State(TypedDict):
    messages: list[AnyMessage]
    extra_field: int

定义节点
****

In [7]:
from langchain_core.messages import AIMessage

def node(state: State):
    messages = state['messages']
    new_messages = AIMessage('你好!我是节点1')
    return {
        "messages": messages + [new_messages],
        "extra_field": 1
    }

创建图
****
- 包含一个节点
- 使用state通信

In [ ]:
from langgraph.graph import StateGraph

graph = StateGraph(State)
graph.add_node(node)
graph.set_entry_point("node")
graph_builder = graph.build()

查看节点与图结构

In [ ]:
from IPython.display import display, Image
display(Image(graph_builder.get_graph().draw_mermaid_png()))

调用

In [ ]:
from langchain_core.message import HumanMessage

result = graph_builder.invoke({
    "messages": [HumanMessage("你好!节点1")]
})

result

使用pretty_print来格式化显示
****

In [ ]:
from langchain_core.messages import HumanMessage

result = graph_builder.invoker({
    "messages": [HumanMessage("你好!节点1")]
})
for message in result['messages']:
    message.pretty_print()

串行控制
****

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    value_1: str
    value_2: str

def step_1(state: State):
    return {
        "value_1": "a"
    }

def step_2(state: State):
    current_value_1 = state['value_1']
    return { "value_1": f"{current_value_1}+b" }

def step_3(state: State):
    return { "value2": 10 }

graph_builder = StateGraph(State)
graph_builder.add_node(step_1)
graph_builder.add_node(step_2)
graph_builder.add_node(step_3)

graph_builder.add_edge(START, 'step_1')
graph_builder.add_edge('step_1', 'step_2')
graph_builder.add_edge('step_2', 'step_3')
graph_builder.add_edge('step_3', END)

graph = graph_builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))
graph.invoke({ "value_1": "c"})

分支控制
****

In [ ]:
import operator
from typing import Any, Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import display, Image

# Annotated 允许为类型提供额外的元数据，而不影响类型查检器对类型本身的理解
class State(TypedDict):
    aggregate: Annotated[list, operator.add]

def a(state: State): 
    print(f"添加“A”到{state['aggregate']}")
    return {"aggregate": ["A"]}

def b(state: State):
    print(f"添加“B”到{state['aggregate']}")
    return {"aggregate": ["B"]}

def c(state: State):
    print(f"添加“C”到{state['aggregate']}")
    return {"aggregate": ["C"]}

def d(state: State):
    print(f"添加“D”到{state['aggregate']}")
    return {"aggregate": ["D"]}

builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_node(c)
builder.add_node(d)

builder.add_edge(START, 'a')
builder.add_edge('a', 'b')
builder.add_edge('a', 'c')
builder.add_edge('b', 'd')
builder.add_edge('c', 'd')
builder.add_edge('d', END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

条件分支和循环
*** 

In [ ]:
import operator
from typing import Any, Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import display, Image

# Annotated 允许为类型提供额外的元数据，而不影响类型查检器对类型本身的理解
class State(TypedDict):
    aggregate: Annotated[list, operator.add]

def a(state: State): 
    print(f"添加“A”到{state['aggregate']}")
    return {"aggregate": ["A"]}

def b(state: State):
    print(f"添加“B”到{state['aggregate']}")
    return {"aggregate": ["B"]}

# Define nodes
builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)

# Define edges
def route(state: State) -> Literal['b', END]: 
    if len(state["aggregate"]) < 7:
        return "b"
    else:
        return END

builder.add_edge(START, 'a')
builder.add_conditional_edge('a', route)
builder.add_edge('b', 'a')    

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))
graph.invoke({"aggregate": []})
    

图的运行时配置

In [ ]:
import operator
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI
import os
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.runnables import Runnable
from langgraph.graph import StateGraph, START, END


model = ChatDeepSeek(
    model="Pro/deepseek-ai/DeepSeek-v3",
    temperature=0.7,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL") 
)

model1 = ChatOpenAI(
   model="gpt-3.5-turbo",
   temperature=0.7
   api_key='',
   base_url=""
)

# 定义要切换的模型
models = {
    "deepseek": model,
    "openai": model1
}

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

def _call_model(state: AgentState, config: RunnableConfig):
    model_name = config["configurable"].get("model", "deepseek") # 默认使用deepseek
    model = models[model_name]
    response = model.invoke(state["messages"])
    return {"messages": [response]}

# Define a new graph
builder = StateGraph(AgentState)
builder.add_node("model", _call_model)
builder.add_edage(START, "model")
builder.add_edage("model", END)

graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))
        

在没有运行时的情况下默认使用deepseek


In [ ]:
graph.invoke({"messages": [HumanMessage('Hi 你是谁')]})

增加运行时配置，动态切换模型

In [ ]:
config = { 'configurable': { "model": 'openai' } }

graph.invoke({"messages": [HumanMessage('Hi 你是谁')]}, config=config)

map-reduce 精细化控制

In [ ]:
from langchain_deepseek import ChatDeepSeek
from langgraph.types import Send
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

# 模型和提示词
# 定义我们将使用的模型与提示词
subjects_prompt = """生成一个逗号分隔的列表，包含2到5个与以下主题相关的例子: {topic}"""
joke_prompt = """生成一个关于{subject}的笑话"""
best_joke_prompt = """以下是一些关于{topic}的笑话。选出最好的一个！返回最佳笑话的ID

{jokes}"""

class Subjects(BaseModel):
    subjects: list[str]

class Joke(BaseModel):
    joke: str    

class BestJoke(BaseModel):
    id: int = Field(description="最佳笑话的索引，从0开始", ge=0)

model = ChatDeepSeek(
    model="Pro/deepseek-ai/DeepSeek-v3",
    temperature=0.7,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL") 
)

# 这将是主图的整体状态
# 它将包含一个主题（我们希望用户提供）
# 然后将成生一个主题列表，并为每个主题生成一个笑话
class OverallState(TypedDict):
    topic: str
    subject: list
    # 注意这里我们使用operator.add
    # 这是因为我们想把从各个节点生成的笑话
    # 合并回一个列表-这本质上是‘归约’部分
    jokes: Annotated[list, operator.add]
    best_selected_joke: str

# 用户生成笑话
class JokeState(TypedDict):
    subject: str   

# 生成笑话主题的函数
def generate_topics(state: OverallState):
    prompt = subjects_prompt.format(topic=state['topic'])
    response = model.with_structured_output(Subjects).invoke(prompt)
    return { "subjects": response.subjects }

# 生成笑话的函数
def generate_joke(state: OverallState):
    prompt = subjects_prompt.format(topic=state['subject'])
    response = model.with_structured_output(Joke).invoke(prompt)
    return { "jokes": [response.joke] }

# 这里我们定义映射到生成主题上的逻辑
# 我们将在图中使用这个作为边缘
def continue_to_jokes(state: OverallState):
    # 返回一个Send对象列表
    # 每个Send对象包含图中节点的名称
    # 以及要发送到该节点的状态
    return [Send("generate_joke", {"subject": s}) for s in state["subjects"]]

# 这里我们评判最佳笑话
def best_joke(state: OverallState):
    jokes = "\n\n".join(state["jokes"])
    prompt = best_joke_prompt.format(topic=state['topic'], jokes=jokes)
    response = model.with_structured_output(BestJoke).invoke(prompt)
    return { "best_selected_joke": state["jokes"][response.id] }


# 构建图: 这里我们将所有内容组合在一起构建我们的图
graph = StateGraph(OverallState)
graph.add_node("generate_topics", generate_topics)
graph.add_node("generate_joke", generate_joke)
graph.add_node("best_joke", best_joke)

graph.add_edge(START, "generate_topics")
graph.add_conditional_edges("generate_topics", continue_to_jokes, ['generate_joke'])
graph.add_edge("generate_joke", "best_joke")
graph.add_edge("best_joke", END)
app = graph.compile()

In [ ]:
for s in app.stream({"topic": "动物"}):
    print(s)